# Experimentos — chaves randomizadas

Pipeline **completamente isolado** do `experimentos_ordenado.ipynb`. Aqui inserimos
as chaves em **ordem aleatória** (shuffle do CSV de ids 1..N com semente fixa)
para responder duas perguntas:

1. **EXP_R1** — replicar a estrutura do EXP3 ordenado (varia cache, curvas por
   ordem) com dados embaralhados, para comparação direta.
2. **EXP_R2** — comparar tamanhos de cache contra **cache=1** (baseline) e medir
   o **ganho % operacional** (tempo, reads, writes) para entender se aumentar o
   cache ainda traz benefício *sem* o limiar limpo da inserção sequencial.

Tudo escreve em `experimentos_random/`; datasets e árvores randomizados ganham
sufixo `_rand` (em `datasets/` e `arvores/`, gitignorados como os sequenciais).

## 1. Imports e constantes

In [ ]:
import subprocess, os, sys, random
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATASETS     = ROOT / "datasets"
ARVORES      = ROOT / "arvores"
EXPERIMENTOS = ROOT / "experimentos_random"   # pasta DEDICADA ao pipeline random
for d in (DATASETS, ARVORES, EXPERIMENTOS):
    d.mkdir(parents=True, exist_ok=True)

# --- Parametros: mesmos eixos do EXP3 ordenado para permitir comparacao ---
R_SIZE   = 100_000
R_ORDERS = [4, 8, 16]
R_CACHES = [1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 16, 20, 24, 32]   # INCLUI 1 (baseline EXP_R2)

SEARCH_N  = 10_000
SEED      = 42       # semente das queries (igual ao ordenado)
RAND_SEED = 7        # semente do shuffle do CSV (ordem de insercao)

COLS = ["order", "size", "cache", "phase", "n", "time_s", "reads", "writes", "found"]
print("Diretorios prontos.")

## 2. Helpers (build, dataset shuffled, runs, parsing)

In [ ]:
def sh(cmd):
    "Roda comando (lista de args) e devolve stdout; levanta em erro."
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("Falhou: " + " ".join(map(str, cmd)) + "\nstderr:\n" + r.stderr)
    return r.stdout

def ensure_bench(order):
    "Compila bin/bench_<order> se necessario (make idempotente)."
    sh(["make", f"bin/bench_{order}"])
    return f"./bin/bench_{order}"

def ensure_dataset_seq(size):
    "Gera datasets/n<size>.csv (ids sequenciais 1..N) se faltar."
    path = DATASETS / f"n{size}.csv"
    if not path.exists():
        sh([sys.executable, "generate_dataset.py", "-n", size, "-o", path, "-s", SEED])
    return str(path)

def ensure_dataset_random(size):
    "Gera datasets/n<size>_rand.csv (ids embaralhados, semente RAND_SEED)."
    seq = Path(ensure_dataset_seq(size))
    rnd = DATASETS / f"n{size}_rand.csv"
    if rnd.exists():
        return str(rnd)
    with open(seq) as f:
        lines = f.readlines()
    header, body = lines[0], lines[1:]
    random.Random(RAND_SEED).shuffle(body)
    with open(rnd, "w") as f:
        f.write(header); f.writelines(body)
    print(f"Gerado dataset randomizado: {rnd}")
    return str(rnd)

def tree_path_rand(order, size):
    return str(ARVORES / f"o{order}_n{size}_rand.dat")

def parse_result(stdout):
    line = next(l for l in stdout.splitlines() if l.startswith("RESULT"))
    d = dict(tok.split("=") for tok in line.split(",")[1:])
    return {"order": int(d["order"]), "cache": int(d["cache"]), "phase": d["phase"],
            "n": int(d["n"]), "time_s": float(d["time_s"]), "reads": int(d["reads"]),
            "writes": int(d["writes"]), "found": int(d["found"])}

def run_insert_rand(order, size, cache, tree):
    "Insert com chaves embaralhadas; build FRESCO em <tree>."
    for ext in ("", ".meta"):
        p = Path(tree + ext)
        if p.exists(): p.unlink()
    ds = ensure_dataset_random(size)
    bench = ensure_bench(order)
    out = sh([bench, "--phase", "insert", "--tree", tree, "--dataset", ds, "--cache", cache])
    row = parse_result(out); row["size"] = size
    return row

def run_search_rand(order, size, cache, tree, n=SEARCH_N):
    "Search reusa a arvore randomizada ja construida."
    bench = ensure_bench(order)
    out = sh([bench, "--phase", "search", "--tree", tree, "--cache", cache,
              "--n", n, "--n-keys", size, "--seed", SEED])
    row = parse_result(out); row["size"] = size
    return row

def ensure_tree_rand(order, size, build_cache=64):
    "Constroi a arvore randomizada canonica (nao-medicao) se faltar."
    tree = tree_path_rand(order, size)
    if not Path(tree).exists():
        ds = ensure_dataset_random(size)
        bench = ensure_bench(order)
        sh([bench, "--phase", "insert", "--tree", tree, "--dataset", ds, "--cache", build_cache])
    return tree

## 3. Smoke test (rodar PRIMEIRO)

In [ ]:
def smoke():
    o, s, nq = 8, 1000, 200
    tmp = "/tmp/r_smoke.dat"
    ensure_dataset_random(s)
    for c in (2, 10):
        ins = run_insert_rand(o, s, c, tmp)
        assert ins["writes"] > 0
        assert ins["time_s"] > 0
        sch = run_search_rand(o, s, c, tmp, n=nq)
        assert 0 < sch["found"] < nq
    for ext in ("", ".meta"):
        p = Path(tmp + ext)
        if p.exists(): p.unlink()
    print("SMOKE RANDOM OK")

smoke()

## 4. EXP_R1 — pipeline random (espelho do EXP3 ordenado)

Mesma estrutura do EXP3 (varia cache, uma curva por ordem), porém com dataset
embaralhado. A árvore canônica para a busca é construída uma vez por ordem; o
insert é remedido por cache numa árvore descartável (`_tmp_rand_build.dat`).

In [ ]:
tmp = str(ARVORES / "_tmp_rand_build.dat")
rows = []
for o in R_ORDERS:
    canon = ensure_tree_rand(o, R_SIZE)                # canonica p/ busca
    for c in R_CACHES:
        rows.append(run_insert_rand(o, R_SIZE, c, tmp))    # insert medido por cache
        rows.append(run_search_rand(o, R_SIZE, c, canon))  # search reusa canonica
for ext in ("", ".meta"):
    p = Path(tmp + ext)
    if p.exists(): p.unlink()
df_r1 = pd.DataFrame(rows)[COLS]
df_r1.to_csv(EXPERIMENTOS / "exp_r1.csv", index=False)
df_r1

## 5. Gráficos EXP_R1 (writes/n, reads/n, time vs cache)

In [ ]:
def plot_r1(df, outdir, prefix="exp_r1"):
    metrics = ("time_s", "reads", "writes")
    for metric in metrics:
        for phase in ("insert", "search"):
            if metric == "writes" and phase == "search":
                continue  # busca e' read-only -> writes=0 sempre
            sub = df[df["phase"] == phase]
            if sub.empty: continue
            fig, ax = plt.subplots(figsize=(7.5, 4.5))
            orders = sorted(sub["order"].unique())
            for o in orders:
                s = sub[sub["order"] == o].sort_values("cache")
                ax.plot(s["cache"], s[metric], marker="o", label=f"ordem {o}")
            # marcador ORDER = CACHE (mesma referencia visual do sequencial)
            for o in orders:
                ax.axvline(o, color="red", ls="--", lw=1, alpha=0.6)
            ax.plot([], [], color="red", ls="--", lw=1, label="ORDER = CACHE")
            ax.set_xscale("log")
            ax.set_xlabel("cache (nós)"); ax.set_ylabel(metric)
            ax.set_title(f"R1 random: {metric} ({phase})")
            ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
            fig.tight_layout()
            fig.savefig(outdir / f"{prefix}_{metric}_{phase}.png", dpi=120)
            plt.close(fig)

plot_r1(df_r1, EXPERIMENTOS)
print("PNGs R1 salvos em", EXPERIMENTOS)


## 6. EXP_R2 — ganho % vs cache=1

Para cada ordem, fixa `cache=1` como baseline e calcula o ganho percentual de
cada cache > 1 em **tempo**, **reads** e **writes**:

```
ganho% = (metrica_cache1 − metrica_cacheC) / metrica_cache1 × 100
```

Valores positivos = melhora; negativos = piora. Salva tabela e gráficos.

In [ ]:
# Baseline (cache=1) por (order, phase)
base = (df_r1[df_r1["cache"] == 1]
        [["order", "phase", "time_s", "reads", "writes"]]
        .rename(columns={"time_s": "time_s_base",
                         "reads":  "reads_base",
                         "writes": "writes_base"}))

g = df_r1.merge(base, on=["order", "phase"])
for m in ("time_s", "reads", "writes"):
    denom = g[f"{m}_base"].replace(0, pd.NA)
    g[f"{m}_gain_pct"] = (g[f"{m}_base"] - g[m]) / denom * 100
g = g[g["cache"] > 1].copy()

cols_out = ["order", "cache", "phase", "n",
            "time_s", "time_s_base", "time_s_gain_pct",
            "reads",  "reads_base",  "reads_gain_pct",
            "writes", "writes_base", "writes_gain_pct"]
g[cols_out].to_csv(EXPERIMENTOS / "exp_r2_gains.csv", index=False)

# Graficos: ganho % vs cache; curvas por ordem; marcador ORDER=CACHE
for metric in ("time_s", "reads", "writes"):
    for phase in ("insert", "search"):
        if metric == "writes" and phase == "search":
            continue
        sub = g[g["phase"] == phase]
        if sub.empty: continue
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        orders = sorted(sub["order"].unique())
        for o in orders:
            s = sub[sub["order"] == o].sort_values("cache")
            ax.plot(s["cache"], s[f"{metric}_gain_pct"], marker="o", label=f"ordem {o}")
        for o in orders:
            ax.axvline(o, color="red", ls="--", lw=1, alpha=0.6)
        ax.plot([], [], color="red", ls="--", lw=1, label="ORDER = CACHE")
        ax.axhline(0, color="gray", lw=0.8)
        ax.set_xscale("log")
        ax.set_xlabel("cache (nós)")
        ax.set_ylabel(f"ganho % em {metric} (vs cache=1)")
        ax.set_title(f"R2 random: ganho % de {metric} ({phase})")
        ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(EXPERIMENTOS / f"exp_r2_gain_{metric}_{phase}.png", dpi=120)
        plt.close(fig)

print("Tabela + PNGs R2 salvos em", EXPERIMENTOS)
g[["order", "cache", "phase", "time_s_gain_pct", "reads_gain_pct", "writes_gain_pct"]]
